# Data Lab 5 — Calculating OEE from a Production Log
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

Uses a shared synthetic **factory production log** — one row per part made across 3 presses, 2 shifts, 5 days. pandas, NumPy and matplotlib are pre-installed in Colab, so there is nothing to install.

## Objectives
By the end you will be able to:
- Merge two tables on shared keys.
- Aggregate counts per shift and machine.
- Compute Availability x Performance x Quality = OEE and rank machines.

## Load both tables
The production log (one row per part) and a shift summary (planned time & downtime).

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
prod = pd.read_csv(DATA + "production_log.csv")
summ = pd.read_csv(DATA + "shift_summary.csv")
print(prod.shape, summ.shape)
summ.head()

## 1 · Count total & good parts per shift-machine

In [ ]:
counts = prod.groupby(["date","shift","machine"]).agg(
    total=("result", "size"),
    good=("result", lambda s: (s == "Pass").sum()),
).reset_index()
counts.head()

## 2 · Merge with the shift summary
A join on the shared keys `date, shift, machine`.

In [ ]:
oee = counts.merge(summ, on=["date","shift","machine"])
oee["run_min"] = oee["planned_minutes"] - oee["downtime_minutes"]
oee.head()

## 3 · Availability x Performance x Quality = OEE
- **Availability** = run time / planned time
- **Performance** = (parts x ideal cycle) / run time
- **Quality** = good parts / total parts

In [ ]:
oee["Availability"] = oee["run_min"] / oee["planned_minutes"]
oee["Performance"]  = (oee["total"] * oee["ideal_cycle_s"] / 60) / oee["run_min"]
oee["Performance"]  = oee["Performance"].clip(upper=1.0)
oee["Quality"]      = oee["good"] / oee["total"]
oee["OEE"]          = oee["Availability"] * oee["Performance"] * oee["Quality"]
oee[["date","shift","machine","Availability","Performance","Quality","OEE"]].round(3).head()

## 4 · Rank the machines

In [ ]:
summary = oee.groupby("machine")[["Availability","Performance","Quality","OEE"]].mean().round(3)
print(summary.sort_values("OEE"))

## Debrief
- Which machine has the lowest OEE, and which of the three factors drags it down most?
- OEE turns a whole log into one number an owner can act on — the top of the analytics ladder from the console.